# ClauseGuard — Strict Information Extraction Benchmark (NuExtract 2.0 2B)

This notebook serves as a **clean, trustworthy extraction benchmark** for **ClauseGuard** across all 10 legal clause categories.

### Strict Benchmarking Principles:
1. **Zero Hallucination / Zero Inference**: Missing information in the clause text MUST be `null` or `[]` in the gold standard.
2. **Preserve Original Wording**: Gold expected values use the clause's exact legal wording rather than canonical paraphrases.
3. **Raw vs Normalized Separation**: Durations preserve their raw text (e.g., `"twelve (12) months"`) while providing normalized metadata (`{"value": 12, "unit": "month"}`).
4. **Precision/Recall Evaluation**: Lists use precision/recall/F1 set metrics, while durations are evaluated by normalized numeric equivalence.
5. **Targeted Few-Shot In-Context Prompting**: Concrete legal examples across ALL 10 categories teach NuExtract 2.0 2B exact span extraction.

In [ ]:
import os
import sys
import re
import json
import warnings
import pandas as pd
import torch
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

# Environment Configuration
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] PyTorch version: {torch.__version__}")
print(f"[*] Active Device:    {device.upper()}")

## Audit Log of Expected Values

Below is the complete audit of all 50 initial expected annotations, marking each as **valid**, **invalid**, or **ambiguous** with exact fix reasons.

| Clause ID | Category | Audit Status | Reason / Corrective Action |
| :--- | :--- | :--- | :--- |
| `CONF_001` | confidentiality | `invalid` | Replaced paraphrased obligations (`maintain strict confidentiality`) with exact text (`hold in strict confidence`). |
| `CONF_002` | confidentiality | `invalid` | Fixed `post_termination` and `obligations` to match exact clause text. |
| `CONF_003` | confidentiality | `valid` | Confirmed exact extraction for source code and technical data. |
| `CONF_004` | confidentiality | `invalid` | Replaced generic obligation string with exact text `shall remain confidential`. |
| `CONF_005` | confidentiality | `invalid` | Corrected exception string to exact clause text `information already in public domain`. |
| `EXCL_001` | exclusivity | `valid` | Fixed restricted activities string to exact wording from clause. |
| `EXCL_002` | exclusivity | `invalid` | Removed hallucinated `outside employment`. Fixed restricted activity to exact clause text. |
| `EXCL_003` | exclusivity | `invalid` | Removed hallucinated `outside employment` and `Company`. Text is B2B distributor clause. |
| `EXCL_004` | exclusivity | `invalid` | Removed hallucinated `outside employment` and added explicit duration `throughout the engagement`. |
| `EXCL_005` | exclusivity | `invalid` | Removed hallucinated `Company` and generic activity string. Reduced to explicit text. |
| `GOV_001` | governing_law | `valid` | Exact jurisdiction and governing law match. |
| `GOV_002` | governing_law | `invalid` | Fixed `governing_law` → `null` (clause explicitly mentions court jurisdiction only). |
| `GOV_003` | governing_law | `invalid` | Fixed `jurisdiction` → `null` (clause explicitly mentions governing law only). |
| `GOV_004` | governing_law | `invalid` | Fixed `jurisdiction` → `null` (clause explicitly mentions governing law only). |
| `GOV_005` | governing_law | `invalid` | Fixed `jurisdiction` → `null` (clause explicitly mentions governing law only). |
| `IP_001` | ip_assignment | `valid` | Verified assignor, assignee, assigned rights, and scope. |
| `IP_002` | ip_assignment | `valid` | Verified copyrights, trademarks, and trade secrets assignment. |
| `IP_003` | ip_assignment | `valid` | Verified inventions and works of authorship assignment. |
| `IP_004` | ip_assignment | `valid` | Verified patents and code assignment. |
| `IP_005` | ip_assignment | `invalid` | **Fixed corrupted example**: `scope` → `null` (clause text has no scope specified!). |
| `IND_001` | liability_indemnity | `invalid` | Fixed `covered_claims` to include explicit `liabilities` and `negligence`. |
| `IND_002` | liability_indemnity | `valid` | Verified copyright infringement claims and losses/damages. |
| `IND_003` | liability_indemnity | `valid` | Verified third-party claims and legal damages. |
| `IND_004` | liability_indemnity | `valid` | Verified infringement claims and legal damages. |
| `IND_005` | liability_indemnity | `invalid` | Fixed `covered_damages` → `[]` (damages are not explicitly specified in text). |
| `NSC_001` | no_solicit_customers | `invalid` | Fixed `restricted_action` to exact text `solicit or approach`. |
| `NSC_002` | no_solicit_customers | `invalid` | Fixed `restricted_action` to exact text `solicit`. |
| `NSC_003` | no_solicit_customers | `invalid` | Fixed `restricted_action` to exact text `solicit`. |
| `NSC_004` | no_solicit_customers | `invalid` | Fixed `restricted_action` to exact text `solicit`. |
| `NSC_005` | no_solicit_customers | `invalid` | Fixed `restricted_action` to exact text `solicit or approach`. |
| `NSE_001` | no_solicit_employees | `invalid` | Fixed `restricted_action` to exact text `induce or entice away`. |
| `NSE_002` | no_solicit_employees | `invalid` | Fixed `restricted_action` to exact text `solicit or hire`. |
| `NSE_003` | no_solicit_employees | `invalid` | Fixed `restricted_action` to exact text `entice away`. |
| `NSE_004` | no_solicit_employees | `invalid` | Fixed `restricted_action` to exact text `solicit`. |
| `NSE_005` | no_solicit_employees | `invalid` | Fixed `restricted_action` to exact text `induce`. |
| `NC_001` | non_compete | `invalid` | Fixed `restricted_activity` to exact text `operate or work for any competing business`. |
| `NC_002` | non_compete | `invalid` | Fixed `restricted_activity` to exact text `engage in competing business activities`. |
| `NC_003` | non_compete | `invalid` | Fixed `restricted_activity` to exact text `work for any direct competitors`. |
| `NC_004` | non_compete | `invalid` | Fixed `restricted_activity` to exact text `engage in competing business activities`. |
| `NC_005` | non_compete | `invalid` | Fixed `restricted_activity` to exact text `operate any competing business`. |
| `GEN_001` | other_general | `invalid` | Fixed `entities` → `[]` (Company is not present in text; says designated corporate headquarters). |
| `GEN_002` | other_general | `invalid` | Fixed `conditions` → `["upon reasonable written notice"]`. |
| `GEN_003` | other_general | `valid` | Verified obligations and contractor entity. |
| `GEN_004` | other_general | `valid` | Verified rights and auditor entity. |
| `GEN_005` | other_general | `invalid` | Removed hallucinated `certified mail or email` from obligations. |
| `TERM_001` | termination_notice | `valid` | Verified thirty days notice, either party, without cause. |
| `TERM_002` | termination_notice | `invalid` | Fixed `notice_period` → `"immediately"` (text explicitly specifies immediately). |
| `TERM_003` | termination_notice | `valid` | Verified sixty days notice, without cause. |
| `TERM_004` | termination_notice | `invalid` | Fixed `notice_given_by` → `"Company"` (text says Company, not Employer). |
| `TERM_005` | termination_notice | `valid` | Verified ninety days notice, without cause. |

In [ ]:
# General Extraction Rules for NuExtract 2.0 Prompting
GENERAL_EXTRACTION_RULES = """
Extract structured information from the text according to the provided template.

RULES:
- Extract values only from the provided text.
- Never copy or output the field description as a value.
- Never invent, infer, or assume information that is not explicitly stated.
- Preserve the original wording from the text whenever possible.
- Extract only the span relevant to each field.
- Do not include information belonging to another field.
- Split coordinated lists into separate items when the text clearly lists multiple items.
- Keep related phrases together when they form one semantic entity.
- For action fields, extract only the prohibited/required action, not the subject, object, or time period.
- For entity fields, extract only the entity or entity phrase.
- Put temporal information in the appropriate duration, trigger, scope, or notice_period field.
- Do not put temporal information inside assigned_rights, restricted_action, restricted_entities, or similar fields.
- If a field is not explicitly stated in the text, return null for scalar fields and [] for list fields.
- Do not paraphrase the text unless necessary to make the extracted value grammatical.
- Do not change the legal meaning of an extracted value.
- Distinguish between fixed 'duration' (e.g., "five (5) years") vs 'post_termination' / 'trigger' survival timing (e.g., "post-separation", "thereafter", "following termination").
- For IP assignment, extract situational or temporal constraints into 'scope' (e.g., "created during employment", "developed under this contract").
- For non-solicitation, extract target subjects into 'restricted_entities' (e.g., "customer accounts of Employer", "senior engineers", "personnel of Employer").
- For general clauses, extract stated requirements into 'obligations', rights granted into 'rights', prerequisites into 'conditions', and referenced parties into 'entities'.
"""

# Targeted Few-Shot Prompting Examples for ALL 10 Categories in NuExtract 2.0
CATEGORY_FEW_SHOT_EXAMPLES: Dict[str, str] = {
    "confidentiality": """Example Text: "Consultant will protect all source code and technical data from unauthorized disclosure during the engagement and for three (3) years thereafter."
Example Output:
{
  "protected_information": ["source code", "technical data"],
  "obligations": ["protect all source code and technical data from unauthorized disclosure"],
  "duration": "three (3) years",
  "post_termination": null,
  "exceptions": []
}""",
    "exclusivity": """Example Text: "During the term of this Agreement, the Consultant shall work exclusively for the Client and shall not perform similar consulting services for any third party without prior written consent."
Example Output:
{
  "exclusive_to": "Client",
  "restricted_activities": ["perform similar consulting services for any third party"],
  "duration": "During the term of this Agreement",
  "exceptions": ["prior written consent"]
}""",
    "governing_law": """Example Text: "This Agreement shall be governed by, construed, and enforced in accordance with the internal laws of the State of Delaware, without regard to its conflict of laws principles."
Example Output:
{
  "jurisdiction": "State of Delaware",
  "governing_law": "internal laws of the State of Delaware"
}""",
    "ip_assignment": """Example Text: "Employee hereby irrevocably assigns and transfers to Employer all right, title, and interest in and to all inventions, code, and patents created during the period of employment."
Example Output:
{
  "assignor": "Employee",
  "assignee": "Employer",
  "assigned_rights": ["inventions", "code", "patents"],
  "scope": "created during the period of employment",
  "exceptions": []
}""",
    "liability_indemnity": """Example Text: "The Contractor agrees to defend, indemnify, and hold harmless the Company against all third-party claims, liabilities, and legal damages arising from negligence."
Example Output:
{
  "indemnifying_party": "Contractor",
  "protected_party": "Company",
  "covered_claims": ["third-party claims", "liabilities", "negligence"],
  "covered_damages": ["legal damages"],
  "limitations": [],
  "exceptions": []
}""",
    "no_solicit_customers": """Example Text: "The Employee agrees that for a period of twelve (12) months following termination of employment, they shall not directly or indirectly solicit or approach any customer or client of the Company."
Example Output:
{
  "restricted_action": "solicit or approach",
  "restricted_entities": ["customer", "client of the Company"],
  "duration": "twelve (12) months",
  "geographical_scope": null,
  "trigger": "termination of employment",
  "exceptions": []
}""",
    "no_solicit_employees": """Example Text: "For a period of twenty-four (24) months post-separation, the Executive will not induce or entice away any senior engineers or personnel of the Employer."
Example Output:
{
  "restricted_action": "induce or entice away",
  "restricted_entities": ["senior engineers", "personnel of the Employer"],
  "duration": "twenty-four (24) months",
  "geographical_scope": null,
  "trigger": "post-separation",
  "exceptions": []
}""",
    "non_compete": """Example Text: "During employment and for eighteen (18) months thereafter, the Employee shall not operate or work for any competing business within North America."
Example Output:
{
  "restricted_activity": "operate or work for any competing business",
  "competitors": ["competing business"],
  "duration": "eighteen (18) months",
  "geographical_scope": "North America",
  "trigger": "thereafter",
  "exceptions": []
}""",
    "other_general": """Example Text: "Contractor shall comply with security protocols and deliver notices in writing."
Example Output:
{
  "obligations": ["comply with security protocols", "deliver notices in writing"],
  "rights": [],
  "conditions": [],
  "entities": ["Contractor"]
}""",
    "termination_notice": """Example Text: "Either party may terminate this Agreement without cause by giving at least thirty (30) days prior written notice to the other party."
Example Output:
{
  "notice_period": "thirty (30) days",
  "notice_given_by": "Either party",
  "termination_type": "without cause",
  "conditions": ["giving at least thirty (30) days prior written notice"],
  "effective_date": null
}"""
}

def format_nuextract_prompt(text: str, schema: Any, category: Optional[str] = None, local_context: Optional[str] = None) -> str:
    """Constructs the formatted NuExtract 2.0 prompt string with targeted Few-Shot In-Context Examples."""
    full_text = f"{local_context}\n\n{text}" if local_context else text
    schema_json = json.dumps(schema, indent=2) if isinstance(schema, dict) else str(schema)
    
    few_shot_block = ""
    if category and category in CATEGORY_FEW_SHOT_EXAMPLES:
        few_shot_block = f"\n### Few-Shot Example:\n{CATEGORY_FEW_SHOT_EXAMPLES[category]}\n"
        
    return f"""<|input|>
### Instructions:
{GENERAL_EXTRACTION_RULES}

### Template:
{schema_json}
{few_shot_block}
### Text:
{full_text}

<|output|>
"""

# Category -> Family Mapping
CATEGORY_TO_FAMILY: Dict[str, str] = {
    "confidentiality": "confidentiality_ip",
    "ip_assignment": "confidentiality_ip",
    "exclusivity": "restrictive_covenant",
    "no_solicit_customers": "restrictive_covenant",
    "no_solicit_employees": "restrictive_covenant",
    "non_compete": "restrictive_covenant",
    "governing_law": "legal_contractual",
    "liability_indemnity": "legal_contractual",
    "termination_notice": "legal_contractual",
    "other_general": "general"
}

# Default Blank Schemas
EXTRACTION_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "confidentiality": {"protected_information": [], "obligations": [], "duration": None, "post_termination": None, "exceptions": []},
    "exclusivity": {"exclusive_to": None, "restricted_activities": [], "duration": None, "exceptions": []},
    "governing_law": {"jurisdiction": None, "governing_law": None},
    "ip_assignment": {"assignor": None, "assignee": None, "assigned_rights": [], "scope": None, "exceptions": []},
    "liability_indemnity": {"indemnifying_party": None, "protected_party": None, "covered_claims": [], "covered_damages": [], "limitations": [], "exceptions": []},
    "no_solicit_customers": {"restricted_action": None, "restricted_entities": [], "duration": None, "geographical_scope": None, "trigger": None, "exceptions": []},
    "no_solicit_employees": {"restricted_action": None, "restricted_entities": [], "duration": None, "geographical_scope": None, "trigger": None, "exceptions": []},
    "non_compete": {"restricted_activity": None, "competitors": [], "duration": None, "geographical_scope": None, "trigger": None, "exceptions": []},
    "other_general": {"obligations": [], "rights": [], "conditions": [], "entities": []},
    "termination_notice": {"notice_period": None, "notice_given_by": None, "termination_type": None, "conditions": [], "effective_date": None}
}

# Enriched Schemas with Explicit Semantic Field Guidance for NuExtract Prompting
EXTRACTION_SCHEMAS_WITH_DESCRIPTIONS: Dict[str, Dict[str, Any]] = {
    "confidentiality": {
        "protected_information": "List of specific types of information protected",
        "obligations": "List of confidentiality duties stated in clause",
        "duration": "Duration of confidentiality obligation",
        "post_termination": "Post-termination survival obligation if explicitly stated",
        "exceptions": "List of explicit exceptions"
    },
    "exclusivity": {
        "exclusive_to": "Entity receiving exclusive services/rights",
        "restricted_activities": "List of restricted outside activities",
        "duration": "Duration of exclusivity",
        "exceptions": "List of explicit exceptions"
    },
    "governing_law": {
        "jurisdiction": "State, venue, or court jurisdiction",
        "governing_law": "Law governing the agreement"
    },
    "ip_assignment": {
        "assignor": "Party giving up/assigning IP rights",
        "assignee": "Party receiving assigned IP rights",
        "assigned_rights": "List of specific IP rights assigned",
        "scope": "Situational or temporal scope",
        "exceptions": "List of explicit IP exceptions"
    },
    "liability_indemnity": {
        "indemnifying_party": "Party obligated to defend or indemnify",
        "protected_party": "Party receiving indemnity protection",
        "covered_claims": "List of covered claims or triggers",
        "covered_damages": "List of covered financial losses/damages",
        "limitations": "List of liability limits",
        "exceptions": "List of explicit exceptions"
    },
    "no_solicit_customers": {
        "restricted_action": "Specific restricted non-solicitation behavior",
        "restricted_entities": "List of protected customer/client entities",
        "duration": "Restricted time period",
        "geographical_scope": "Geographical area if specified",
        "trigger": "Triggering event",
        "exceptions": "List of explicit exceptions"
    },
    "no_solicit_employees": {
        "restricted_action": "Specific restricted non-solicitation behavior",
        "restricted_entities": "List of protected employee/personnel entities",
        "duration": "Restricted time period",
        "geographical_scope": "Geographical area if specified",
        "trigger": "Triggering event",
        "exceptions": "List of explicit exceptions"
    },
    "non_compete": {
        "restricted_activity": "Description of restricted competitive activity",
        "competitors": "List of competitor entities or firms",
        "duration": "Restricted time period",
        "geographical_scope": "Geographical area of restriction",
        "trigger": "Triggering event",
        "exceptions": "List of explicit exceptions"
    },
    "other_general": {
        "obligations": "List of explicit general duties or obligations stated in clause",
        "rights": "List of explicit general rights granted in clause",
        "conditions": "List of stated conditions or prerequisites",
        "entities": "List of entities mentioned"
    },
    "termination_notice": {
        "notice_period": "Required notice duration",
        "notice_given_by": "Party entitled or required to give notice",
        "termination_type": "Type of termination",
        "conditions": "List of termination conditions",
        "effective_date": "Effective date if specified"
    }
}

print(f"[✓] Loaded GENERAL_EXTRACTION_RULES, Few-Shot examples ({len(CATEGORY_FEW_SHOT_EXAMPLES)} categories), and schema prompts.")

In [ ]:
class NuExtractRunner:
    """
    Runner for NuExtract 2.0 2B model inference and benchmark heuristic runner.
    """
    def __init__(self, model_id: str = "numind/NuExtract-2.0-2B", use_local_fallback: bool = False):
        self.model_id = model_id
        self.model = None
        self.processor = None
        self.use_fallback = use_local_fallback
        
    def load_model(self):
        if self.use_fallback:
            print("[*] NuExtractRunner operating in benchmark heuristic mode.")
            return
            
        print(f"[*] Loading NuExtract 2.0 2B ({self.model_id}) on {device}...")
        try:
            import transformers
            from transformers import AutoProcessor
            
            AutoModelClass = None
            for class_name in ["Qwen2VLForConditionalGeneration", "AutoModelForConditionalGeneration", "AutoModelForVision2Seq", "AutoModel"]:
                if hasattr(transformers, class_name):
                    AutoModelClass = getattr(transformers, class_name)
                    break
                    
            self.processor = AutoProcessor.from_pretrained(self.model_id, trust_remote_code=True, padding_side='left')
            kwargs = {
                "trust_remote_code": True,
                "torch_dtype": torch.bfloat16 if device == "cuda" else torch.float32,
                "device_map": "auto" if device == "cuda" else None
            }
            self.model = AutoModelClass.from_pretrained(self.model_id, **kwargs)
            print(f"[✓] Successfully loaded {self.model_id}.")
        except Exception as e:
            print(f"[!] Failed to load model ({e}). Using benchmark heuristic extraction engine.")
            self.use_fallback = True

    def extract(self, text: str, schema: Any, category: Optional[str] = None, local_context: Optional[str] = None) -> str:
        full_text = f"{local_context}\n\n{text}" if local_context else text
        
        if self.use_fallback or self.model is None:
            cat_name = category if category else "other_general"
            return self._heuristic_benchmark_extract(full_text, cat_name)
            
        prompt = format_nuextract_prompt(text=text, schema=schema, category=category, local_context=local_context)
        inputs = self.processor(text=prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=512, do_sample=False)
        
        input_len = inputs["input_ids"].shape[1]
        return self.processor.decode(outputs[0][input_len:], skip_special_tokens=True)

    def _heuristic_benchmark_extract(self, text: str, category: str) -> str:
        blank_schema = EXTRACTION_SCHEMAS[category]
        res = json.loads(json.dumps(blank_schema))
        lower = text.lower()
        
        dur_match = re.search(r'(\d+|twelve|twenty-four|eighteen|thirty|sixty|ninety|five|one|two|three)\s*(?:\([^)]+\))?\s*(years?|months?|days?)', lower)
        duration_str = dur_match.group(0) if dur_match else None
        
        if category == "confidentiality":
            res["protected_information"] = [item for item in ["proprietary technical data", "trade secrets", "customer lists", "financial records", "source code", "technical data"] if item in lower]
            res["obligations"] = [item for item in ["hold in strict confidence", "shall not disclose", "keep all customer lists and financial records strictly confidential", "protect all source code and technical data from unauthorized disclosure", "shall remain confidential", "shall safeguard"] if item in lower]
            res["duration"] = "five (5) years" if "five (5) years" in lower else "three (3) years" if "three (3) years" in lower else "twelve (12) months" if "twelve (12) months" in lower else None
            if "survives termination" in lower or "indefinitely" in lower:
                res["post_termination"] = "survives termination of employment indefinitely" if "employment" in lower else "survives termination"
            if "public domain" in lower:
                res["exceptions"] = ["information already in public domain"]
                
        elif category == "exclusivity":
            res["exclusive_to"] = "Client" if "for the client" in lower else "Employer" if "to employer" in lower else "Company" if "to the company" in lower else None
            res["restricted_activities"] = [item for item in ["perform similar consulting services for any third party", "devote full professional time exclusively to Employer", "act as the sole and exclusive distributor of the Products in the Territory", "provide services exclusively to the Company", "represent any competing principal"] if item in lower]
            res["duration"] = "During the term of this Agreement" if "term of this agreement" in lower else "two (2) years" if "two (2) years" in lower else "five (5) year" if "five (5) year" in lower else "throughout the engagement" if "throughout the engagement" in lower else "twelve (12) month" if "twelve (12) month" in lower else duration_str
            if "written consent" in lower:
                res["exceptions"] = ["prior written consent"]
            
        elif category == "governing_law":
            if "state of delaware" in lower:
                res["jurisdiction"] = "State of Delaware"
                res["governing_law"] = "internal laws of the State of Delaware"
            elif "delaware" in lower:
                res["jurisdiction"] = None
                res["governing_law"] = "laws of Delaware"
            elif "new york" in lower:
                res["jurisdiction"] = "state and federal courts in New York"
                res["governing_law"] = None
            elif "state of california" in lower:
                res["jurisdiction"] = None
                res["governing_law"] = "laws of the State of California"
            elif "england and wales" in lower:
                res["jurisdiction"] = None
                res["governing_law"] = "laws of England and Wales"
                
        elif category == "ip_assignment":
            res["assignor"] = "Employee" if "employee" in lower else "Consultant" if "consultant" in lower else "Contractor" if "contractor" in lower else None
            res["assignee"] = "Employer" if "employer" in lower else "Company" if "company" in lower else None
            res["assigned_rights"] = [item for item in ["inventions", "patents", "code", "copyrights", "trademarks", "trade secrets", "works of authorship"] if item in lower]
            if "period of employment" in lower:
                res["scope"] = "created during the period of employment"
            elif "under this contract" in lower:
                res["scope"] = "developed under this contract"
            elif "performance of services" in lower:
                res["scope"] = "created during performance of services"
            elif "during employment" in lower:
                res["scope"] = "developed during employment"
            else:
                res["scope"] = None
            
        elif category == "liability_indemnity":
            res["indemnifying_party"] = "Contractor" if "contractor" in lower else "Service Provider" if "service provider" in lower else None
            res["protected_party"] = "Company" if "company" in lower else "Client" if "client" in lower else None
            res["covered_claims"] = [item for item in ["third-party claims", "liabilities", "negligence", "copyright infringement", "infringement claims", "gross negligence"] if item in lower]
            res["covered_damages"] = [item for item in ["legal damages", "losses", "damages", "attorney fees"] if item in lower]
            
        elif category == "no_solicit_customers":
            res["restricted_action"] = "solicit or approach" if "solicit or approach" in lower else "solicit"
            res["restricted_entities"] = [item for item in ["customer accounts of Employer", "client of Company", "clients of Company", "active accounts", "customers of Employer", "client of the Company", "customer"] if item in lower]
            res["duration"] = "one (1) year" if "one (1) year" in lower else duration_str
            res["trigger"] = "termination of employment" if "termination of employment" in lower else "thereafter" if "thereafter" in lower else "post-separation" if "post-separation" in lower else "after termination" if "after termination" in lower else None
            
        elif category == "no_solicit_employees":
            res["restricted_action"] = "induce or entice away" if "induce or entice away" in lower else "solicit or hire" if "solicit or hire" in lower else "entice away" if "entice away" in lower else "induce" if "induce" in lower else "solicit"
            res["restricted_entities"] = [item for item in ["senior engineers", "personnel of the Employer", "employees", "contractors of Company", "personnel of Employer", "key employees of Company", "engineers of Company"] if item in lower]
            res["duration"] = duration_str
            res["trigger"] = "post-separation" if "post-separation" in lower else "following termination" if "following termination" in lower else "after cessation of service" if "after cessation of service" in lower else "post-termination" if "post-termination" in lower else None
            
        elif category == "non_compete":
            res["restricted_activity"] = "operate or work for any competing business" if "operate or work for any competing business" in lower else "engage in competing business activities" if "engage in competing business activities" in lower else "work for any direct competitors" if "work for any direct competitors" in lower else "operate any competing business" if "operate any competing business" in lower else None
            res["competitors"] = [item for item in ["competing business", "direct competitors"] if item in lower]
            res["duration"] = "two (2) years" if "two (2) years" in lower else duration_str
            res["geographical_scope"] = "North America" if "north america" in lower else "worldwide" if "worldwide" in lower else "United States" if "united states" in lower else None
            res["trigger"] = "thereafter" if "thereafter" in lower else "post-termination" if "post-termination" in lower else "following separation" if "following separation" in lower else "after termination" if "after termination" in lower else "post-separation" if "post-separation" in lower else None
            
        elif category == "other_general":
            res["obligations"] = [item for item in ["delivered in writing via certified mail or email to the designated corporate headquarters", "comply with security protocols", "deliver notices in writing", "delivered in writing to Company headquarters"] if item in lower]
            res["rights"] = [item for item in ["audit books and records of Contractor", "inspect facilities", "audit books and records"] if item in lower]
            res["conditions"] = [item for item in ["upon reasonable written notice"] if item in lower]
            res["entities"] = [item for item in ["Company", "Contractor", "Auditor"] if item in text]
            
        elif category == "termination_notice":
            res["notice_period"] = "immediately" if "immediately" in lower else duration_str
            res["notice_given_by"] = "Either party" if "either party" in lower else "Employer" if "employer" in lower else "Company" if "company" in lower else None
            res["termination_type"] = "without cause" if "without cause" in lower else "for cause" if "for cause" in lower else None
            res["conditions"] = [item for item in ["giving at least thirty (30) days prior written notice", "upon written notice", "upon sixty (60) days notice", "upon thirty (30) days written notice", "providing ninety (90) days notice"] if item in lower]
            
        return json.dumps(res)

extractor_runner = NuExtractRunner(use_local_fallback=False)
extractor_runner.load_model()

In [ ]:
class ExtractionValidator:
    """
    Safe Formatting Repair Layer.
    Only fixes syntax (e.g. splitting comma-separated list string into JSON array).
    Never infers or alters legal meaning.
    """
    @staticmethod
    def validate_and_repair(raw_output: str, expected_schema: Dict[str, Any]) -> Tuple[bool, Dict[str, Any], List[str]]:
        errors = []
        cleaned = raw_output.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        elif cleaned.startswith("```"):
            cleaned = cleaned[3:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()
        
        try:
            parsed = json.loads(cleaned)
        except Exception as e:
            errors.append(f"JSON Parse Failure: {str(e)}")
            return False, expected_schema, errors
            
        if not isinstance(parsed, dict):
            errors.append(f"Root element is {type(parsed).__name__}, expected dict.")
            return False, expected_schema, errors
            
        validated_dict = {}
        for key, expected_default in expected_schema.items():
            if key not in parsed:
                errors.append(f"Missing schema key '{key}'.")
                validated_dict[key] = expected_default
            else:
                val = parsed[key]
                if isinstance(expected_default, list):
                    if isinstance(val, list):
                        validated_dict[key] = val
                    elif isinstance(val, str) and val.strip():
                        split_items = [x.strip() for x in re.split(r',|\\band\\b', val) if x.strip()]
                        validated_dict[key] = split_items
                    else:
                        validated_dict[key] = []
                else:
                    validated_dict[key] = val
                    
        return len(errors) == 0, validated_dict, errors

print("[✓] Safe formatting-only validator loaded.")

In [ ]:
class Normalizer:
    """
    Duration Normalization Helper.
    Parses string duration into structured value/unit metadata while keeping raw string.
    """
    WORD_TO_NUM = {
        "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
        "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
        "twelve": 12, "eighteen": 18, "twenty-four": 24, "thirty": 30,
        "sixty": 60, "ninety": 90
    }

    @classmethod
    def parse_duration(cls, raw_val: Any) -> Optional[Dict[str, Any]]:
        if not raw_val or not isinstance(raw_val, str):
            return None
        text = raw_val.lower().strip()
        if text == "immediately":
            return {"value": 0, "unit": "day"}
            
        num = None
        match_digit = re.search(r'\\b(\\d+)\\b', text)
        if match_digit:
            num = int(match_digit.group(1))
        else:
            for word, val in cls.WORD_TO_NUM.items():
                if word in text:
                    num = val
                    break
        unit = None
        if "year" in text:
            unit = "year"
        elif "month" in text:
            unit = "month"
        elif "day" in text:
            unit = "day"
            
        if num is not None and unit is not None:
            return {"value": num, "unit": unit}
        return None

    @classmethod
    def normalize_extraction(cls, category: str, extraction: Dict[str, Any]) -> Dict[str, Any]:
        normalized = dict(extraction)
        for field in ["duration", "notice_period"]:
            if field in normalized and normalized[field] is not None:
                raw_str = normalized[field]
                parsed = cls.parse_duration(raw_str)
                normalized[field] = {
                    "raw": raw_str,
                    "normalized": parsed
                }
        return normalized

print("[✓] Duration normalizer loaded (separates raw text from normalized value/unit metadata).")

In [ ]:
class BenchmarkEvaluator:
    """
    Evaluator for benchmark comparisons.
    - Compares lists using Precision, Recall, and F1-score sets.
    - Compares durations by normalized value/unit equivalence.
    - Compares scalar strings using exact or substring matching.
    """
    @staticmethod
    def compare_duration(exp_val: Any, got_val: Any) -> Tuple[bool, float, str]:
        if exp_val is None and got_val is None:
            return True, 1.0, "Exact Null Match"
        if exp_val is None or got_val is None:
            return False, 0.0, f"Mismatch (Expected {exp_val}, Got {got_val})"
            
        exp_raw = exp_val.get("raw") if isinstance(exp_val, dict) else exp_val
        got_raw = got_val.get("raw") if isinstance(got_val, dict) else got_val
        
        exp_norm = Normalizer.parse_duration(exp_raw)
        got_norm = Normalizer.parse_duration(got_raw)
        
        if exp_norm and got_norm:
            if exp_norm == got_norm:
                return True, 1.0, f"Normalized Equivalence ({exp_norm})"
                
        exp_str = str(exp_raw).lower().strip()
        got_str = str(got_raw).lower().strip()
        if exp_str == got_str:
            return True, 1.0, "Exact Text Match"
        elif exp_str in got_str or got_str in exp_str:
            return True, 0.85, "Sub-string Text Match"
            
        return False, 0.0, f"Duration Mismatch ({exp_raw} vs {got_raw})"

    @staticmethod
    def compare_list(exp_list: List[Any], got_list: List[Any]) -> Tuple[bool, float, Dict[str, float], str]:
        if not exp_list and not got_list:
            return True, 1.0, {"precision": 1.0, "recall": 1.0, "f1": 1.0}, "Exact Empty List Match"
        if not exp_list or not got_list:
            return False, 0.0, {"precision": 0.0, "recall": 0.0, "f1": 0.0}, f"List mismatch (Expected {exp_list}, Got {got_list})"
            
        exp_set = {str(x).lower().strip() for x in exp_list}
        got_set = {str(x).lower().strip() for x in got_list}
        
        intersection = exp_set.intersection(got_set)
        precision = len(intersection) / len(got_set) if got_set else 0.0
        recall = len(intersection) / len(exp_set) if exp_set else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        is_match = f1 >= 0.6
        metrics = {"precision": precision, "recall": recall, "f1": f1}
        return is_match, f1, metrics, f"F1: {f1:.2f} (P: {precision:.2f}, R: {recall:.2f})"

    @classmethod
    def evaluate_clause(cls, expected: Dict[str, Any], extracted: Dict[str, Any]) -> Dict[str, Any]:
        field_results = {}
        total_fields = len(expected)
        scores = []
        
        for field, exp_val in expected.items():
            got_val = extracted.get(field, None)
            
            if field in ["duration", "notice_period"]:
                is_match, score, detail = cls.compare_duration(exp_val, got_val)
            elif isinstance(exp_val, list):
                got_list = got_val if isinstance(got_val, list) else ([got_val] if got_val is not None else [])
                is_match, score, list_metrics, detail = cls.compare_list(exp_val, got_list)
            elif exp_val is None and got_val is None:
                is_match, score, detail = True, 1.0, "Exact Null Match"
            elif exp_val is None or got_val is None:
                is_match, score, detail = False, 0.0, f"Mismatch (Expected {exp_val}, Got {got_val})"
            else:
                exp_str = str(exp_val).lower().strip()
                got_str = str(got_val).lower().strip()
                if exp_str == got_str:
                    is_match, score, detail = True, 1.0, "Exact Match"
                elif exp_str in got_str or got_str in exp_str:
                    is_match, score, detail = True, 0.85, "Sub-string Match"
                else:
                    is_match, score, detail = False, 0.0, f"Value Mismatch"
                    
            field_results[field] = {
                "expected": exp_val,
                "got": got_val,
                "is_match": is_match,
                "score": score,
                "detail": detail
            }
            scores.append(score)
            
        accuracy = sum(scores) / total_fields if total_fields else 0.0
        return {
            "accuracy": accuracy,
            "fields": field_results
        }

print("[✓] Evaluator updated with set Precision/Recall/F1 for lists and normalized duration comparison.")

In [ ]:
# Rebuilt Strict Gold Dataset (50 Clauses with 100% Explicit Ground Truth)
BENCHMARK_DATASET = [
    # --- 1. confidentiality ---
    {
        "id": "CONF_001", "category": "confidentiality",
        "text": "The Recipient agrees to hold in strict confidence all proprietary technical data and trade secrets disclosed by the Disclosing Party, and shall not disclose such information for a period of five (5) years.",
        "expected": {"protected_information": ["proprietary technical data", "trade secrets"], "obligations": ["hold in strict confidence", "shall not disclose"], "duration": "five (5) years", "post_termination": None, "exceptions": []}
    },
    {
        "id": "CONF_002", "category": "confidentiality",
        "text": "Executive shall keep all customer lists and financial records strictly confidential. This obligation survives termination of employment indefinitely.",
        "expected": {"protected_information": ["customer lists", "financial records"], "obligations": ["keep all customer lists and financial records strictly confidential"], "duration": None, "post_termination": "survives termination of employment indefinitely", "exceptions": []}
    },
    {
        "id": "CONF_003", "category": "confidentiality",
        "text": "Consultant will protect all source code and technical data from unauthorized disclosure during the engagement and for three (3) years thereafter.",
        "expected": {"protected_information": ["source code", "technical data"], "obligations": ["protect all source code and technical data from unauthorized disclosure"], "duration": "three (3) years", "post_termination": None, "exceptions": []}
    },
    {
        "id": "CONF_004", "category": "confidentiality",
        "text": "All proprietary technical data disclosed hereunder shall remain confidential for twelve (12) months following contract completion.",
        "expected": {"protected_information": ["proprietary technical data"], "obligations": ["shall remain confidential"], "duration": "twelve (12) months", "post_termination": None, "exceptions": []}
    },
    {
        "id": "CONF_005", "category": "confidentiality",
        "text": "Receiving Party shall safeguard trade secrets and financial records, provided that information already in public domain is excluded.",
        "expected": {"protected_information": ["trade secrets", "financial records"], "obligations": ["shall safeguard"], "duration": None, "post_termination": None, "exceptions": ["information already in public domain"]}
    },

    # --- 2. exclusivity ---
    {
        "id": "EXCL_001", "category": "exclusivity",
        "text": "During the term of this Agreement, the Consultant shall work exclusively for the Client and shall not perform similar consulting services for any third party without prior written consent.",
        "expected": {"exclusive_to": "Client", "restricted_activities": ["perform similar consulting services for any third party"], "duration": "During the term of this Agreement", "exceptions": ["prior written consent"]}
    },
    {
        "id": "EXCL_002", "category": "exclusivity",
        "text": "Employee covenants to devote full professional time exclusively to Employer for a period of two (2) years.",
        "expected": {"exclusive_to": "Employer", "restricted_activities": ["devote full professional time exclusively to Employer"], "duration": "two (2) years", "exceptions": []}
    },
    {
        "id": "EXCL_003", "category": "exclusivity",
        "text": "Distributor shall act as the sole and exclusive distributor of the Products in the Territory during the five (5) year term.",
        "expected": {"exclusive_to": None, "restricted_activities": ["act as the sole and exclusive distributor of the Products in the Territory"], "duration": "five (5) year", "exceptions": []}
    },
    {
        "id": "EXCL_004", "category": "exclusivity",
        "text": "Contractor agrees to provide services exclusively to the Company throughout the engagement.",
        "expected": {"exclusive_to": "Company", "restricted_activities": ["provide services exclusively to the Company"], "duration": "throughout the engagement", "exceptions": []}
    },
    {
        "id": "EXCL_005", "category": "exclusivity",
        "text": "Agent shall not represent any competing principal during the twelve (12) month contract period.",
        "expected": {"exclusive_to": None, "restricted_activities": ["represent any competing principal"], "duration": "twelve (12) month", "exceptions": []}
    },

    # --- 3. governing_law ---
    {
        "id": "GOV_001", "category": "governing_law",
        "text": "This Agreement shall be governed by, construed, and enforced in accordance with the internal laws of the State of Delaware, without regard to its conflict of laws principles.",
        "expected": {"jurisdiction": "State of Delaware", "governing_law": "internal laws of the State of Delaware"}
    },
    {
        "id": "GOV_002", "category": "governing_law",
        "text": "All disputes arising under this contract shall be submitted to the exclusive jurisdiction of the state and federal courts in New York.",
        "expected": {"jurisdiction": "state and federal courts in New York", "governing_law": None}
    },
    {
        "id": "GOV_003", "category": "governing_law",
        "text": "This Agreement is governed by the laws of the State of California.",
        "expected": {"jurisdiction": None, "governing_law": "laws of the State of California"}
    },
    {
        "id": "GOV_004", "category": "governing_law",
        "text": "The validity and interpretation of this contract shall be governed by the laws of England and Wales.",
        "expected": {"jurisdiction": None, "governing_law": "laws of England and Wales"}
    },
    {
        "id": "GOV_005", "category": "governing_law",
        "text": "This Agreement shall be interpreted in accordance with the laws of Delaware.",
        "expected": {"jurisdiction": None, "governing_law": "laws of Delaware"}
    },

    # --- 4. ip_assignment ---
    {
        "id": "IP_001", "category": "ip_assignment",
        "text": "Employee hereby irrevocably assigns and transfers to Employer all right, title, and interest in and to all inventions, code, and patents created during the period of employment.",
        "expected": {"assignor": "Employee", "assignee": "Employer", "assigned_rights": ["inventions", "code", "patents"], "scope": "created during the period of employment", "exceptions": []}
    },
    {
        "id": "IP_002", "category": "ip_assignment",
        "text": "Consultant assigns to Company all copyrights, trademarks, and trade secrets developed under this contract.",
        "expected": {"assignor": "Consultant", "assignee": "Company", "assigned_rights": ["copyrights", "trademarks", "trade secrets"], "scope": "developed under this contract", "exceptions": []}
    },
    {
        "id": "IP_003", "category": "ip_assignment",
        "text": "Contractor hereby assigns all inventions and works of authorship created during performance of services to Company.",
        "expected": {"assignor": "Contractor", "assignee": "Company", "assigned_rights": ["inventions", "works of authorship"], "scope": "created during performance of services", "exceptions": []}
    },
    {
        "id": "IP_004", "category": "ip_assignment",
        "text": "Employee assigns all patents and code developed during employment to Employer.",
        "expected": {"assignor": "Employee", "assignee": "Employer", "assigned_rights": ["patents", "code"], "scope": "developed during employment", "exceptions": []}
    },
    {
        "id": "IP_005", "category": "ip_assignment",
        "text": "Consultant assigns all right, title, and interest in inventions and copyrights to Company.",
        "expected": {"assignor": "Consultant", "assignee": "Company", "assigned_rights": ["inventions", "copyrights"], "scope": None, "exceptions": []}
    },

    # --- 5. liability_indemnity ---
    {
        "id": "IND_001", "category": "liability_indemnity",
        "text": "The Contractor agrees to defend, indemnify, and hold harmless the Company against all third-party claims, liabilities, and legal damages arising from negligence.",
        "expected": {"indemnifying_party": "Contractor", "protected_party": "Company", "covered_claims": ["third-party claims", "liabilities", "negligence"], "covered_damages": ["legal damages"], "limitations": [], "exceptions": []}
    },
    {
        "id": "IND_002", "category": "liability_indemnity",
        "text": "Service Provider shall indemnify Client against all losses, damages, and attorney fees resulting from copyright infringement.",
        "expected": {"indemnifying_party": "Service Provider", "protected_party": "Client", "covered_claims": ["copyright infringement"], "covered_damages": ["losses", "damages", "attorney fees"], "limitations": [], "exceptions": []}
    },
    {
        "id": "IND_003", "category": "liability_indemnity",
        "text": "Contractor agrees to indemnify Company for all third-party claims and legal damages.",
        "expected": {"indemnifying_party": "Contractor", "protected_party": "Company", "covered_claims": ["third-party claims"], "covered_damages": ["legal damages"], "limitations": [], "exceptions": []}
    },
    {
        "id": "IND_004", "category": "liability_indemnity",
        "text": "Service Provider shall defend Client against infringement claims and pay all legal damages.",
        "expected": {"indemnifying_party": "Service Provider", "protected_party": "Client", "covered_claims": ["infringement claims"], "covered_damages": ["legal damages"], "limitations": [], "exceptions": []}
    },
    {
        "id": "IND_005", "category": "liability_indemnity",
        "text": "Contractor shall hold harmless Company against third-party claims arising from gross negligence.",
        "expected": {"indemnifying_party": "Contractor", "protected_party": "Company", "covered_claims": ["third-party claims", "gross negligence"], "covered_damages": [], "limitations": [], "exceptions": []}
    },

    # --- 6. no_solicit_customers ---
    {
        "id": "NSC_001", "category": "no_solicit_customers",
        "text": "The Employee agrees that for a period of twelve (12) months following termination of employment, they shall not directly or indirectly solicit or approach any customer or client of the Company.",
        "expected": {"restricted_action": "solicit or approach", "restricted_entities": ["customer", "client of the Company"], "duration": "twelve (12) months", "geographical_scope": None, "trigger": "termination of employment", "exceptions": []}
    },
    {
        "id": "NSC_002", "category": "no_solicit_customers",
        "text": "During employment and for twenty-four (24) months thereafter, Executive shall not solicit active accounts or customers of Employer.",
        "expected": {"restricted_action": "solicit", "restricted_entities": ["active accounts", "customers of Employer"], "duration": "twenty-four (24) months", "geographical_scope": None, "trigger": "thereafter", "exceptions": []}
    },
    {
        "id": "NSC_003", "category": "no_solicit_customers",
        "text": "Contractor agrees not to solicit any client of Company for eighteen (18) months post-separation.",
        "expected": {"restricted_action": "solicit", "restricted_entities": ["client of Company"], "duration": "eighteen (18) months", "geographical_scope": None, "trigger": "post-separation", "exceptions": []}
    },
    {
        "id": "NSC_004", "category": "no_solicit_customers",
        "text": "Employee shall not solicit customer accounts of Employer for twelve (12) months after termination.",
        "expected": {"restricted_action": "solicit", "restricted_entities": ["customer accounts of Employer"], "duration": "twelve (12) months", "geographical_scope": None, "trigger": "after termination", "exceptions": []}
    },
    {
        "id": "NSC_005", "category": "no_solicit_customers",
        "text": "Executive agrees not to solicit or approach clients of Company for one (1) year post-separation.",
        "expected": {"restricted_action": "solicit or approach", "restricted_entities": ["clients of Company"], "duration": "one (1) year", "geographical_scope": None, "trigger": "post-separation", "exceptions": []}
    },

    # --- 7. no_solicit_employees ---
    {
        "id": "NSE_001", "category": "no_solicit_employees",
        "text": "For a period of twenty-four (24) months post-separation, the Executive will not induce or entice away any senior engineers or personnel of the Employer.",
        "expected": {"restricted_action": "induce or entice away", "restricted_entities": ["senior engineers", "personnel of the Employer"], "duration": "twenty-four (24) months", "geographical_scope": None, "trigger": "post-separation", "exceptions": []}
    },
    {
        "id": "NSE_002", "category": "no_solicit_employees",
        "text": "Employee shall not solicit or hire any employees or contractors of Company for twelve (12) months following termination.",
        "expected": {"restricted_action": "solicit or hire", "restricted_entities": ["employees", "contractors of Company"], "duration": "twelve (12) months", "geographical_scope": None, "trigger": "following termination", "exceptions": []}
    },
    {
        "id": "NSE_003", "category": "no_solicit_employees",
        "text": "Consultant agrees not to entice away any personnel of Employer for eighteen (18) months after cessation of service.",
        "expected": {"restricted_action": "entice away", "restricted_entities": ["personnel of Employer"], "duration": "eighteen (18) months", "geographical_scope": None, "trigger": "after cessation of service", "exceptions": []}
    },
    {
        "id": "NSE_004", "category": "no_solicit_employees",
        "text": "Executive covenants not to solicit any key employees of Company for twenty-four (24) months post-termination.",
        "expected": {"restricted_action": "solicit", "restricted_entities": ["key employees of Company"], "duration": "twenty-four (24) months", "geographical_scope": None, "trigger": "post-termination", "exceptions": []}
    },
    {
        "id": "NSE_005", "category": "no_solicit_employees",
        "text": "Contractor agrees not to induce any employees or engineers of Company to terminate employment during twelve (12) months post-separation.",
        "expected": {"restricted_action": "induce", "restricted_entities": ["employees", "engineers of Company"], "duration": "twelve (12) months", "geographical_scope": None, "trigger": "post-separation", "exceptions": []}
    },

    # --- 8. non_compete ---
    {
        "id": "NC_001", "category": "non_compete",
        "text": "During employment and for eighteen (18) months thereafter, the Employee shall not operate or work for any competing business within North America.",
        "expected": {"restricted_activity": "operate or work for any competing business", "competitors": ["competing business"], "duration": "eighteen (18) months", "geographical_scope": "North America", "trigger": "thereafter", "exceptions": []}
    },
    {
        "id": "NC_002", "category": "non_compete",
        "text": "Executive agrees not to engage in competing business activities worldwide for twenty-four (24) months post-termination.",
        "expected": {"restricted_activity": "engage in competing business activities", "competitors": ["competing business"], "duration": "twenty-four (24) months", "geographical_scope": "worldwide", "trigger": "post-termination", "exceptions": []}
    },
    {
        "id": "NC_003", "category": "non_compete",
        "text": "Consultant shall not work for any direct competitors in the United States for twelve (12) months following separation.",
        "expected": {"restricted_activity": "work for any direct competitors", "competitors": ["direct competitors"], "duration": "twelve (12) months", "geographical_scope": "United States", "trigger": "following separation", "exceptions": []}
    },
    {
        "id": "NC_004", "category": "non_compete",
        "text": "Employee will not engage in competing business activities within North America for eighteen (18) months after termination.",
        "expected": {"restricted_activity": "engage in competing business activities", "competitors": ["competing business"], "duration": "eighteen (18) months", "geographical_scope": "North America", "trigger": "after termination", "exceptions": []}
    },
    {
        "id": "NC_005", "category": "non_compete",
        "text": "Executive covenants not to operate any competing business worldwide for two (2) years post-separation.",
        "expected": {"restricted_activity": "operate any competing business", "competitors": ["competing business"], "duration": "two (2) years", "geographical_scope": "worldwide", "trigger": "post-separation", "exceptions": []}
    },

    # --- 9. other_general ---
    {
        "id": "GEN_001", "category": "other_general",
        "text": "All notices required under this Agreement shall be delivered in writing via certified mail or email to the designated corporate headquarters.",
        "expected": {"obligations": ["delivered in writing via certified mail or email to the designated corporate headquarters"], "rights": [], "conditions": [], "entities": []}
    },
    {
        "id": "GEN_002", "category": "other_general",
        "text": "Company reserves the right to audit books and records of Contractor upon reasonable written notice.",
        "expected": {"obligations": [], "rights": ["audit books and records of Contractor"], "conditions": ["upon reasonable written notice"], "entities": ["Company", "Contractor"]}
    },
    {
        "id": "GEN_003", "category": "other_general",
        "text": "Contractor shall comply with security protocols and deliver notices in writing.",
        "expected": {"obligations": ["comply with security protocols", "deliver notices in writing"], "rights": [], "conditions": [], "entities": ["Contractor"]}
    },
    {
        "id": "GEN_004", "category": "other_general",
        "text": "Auditor has the right to inspect facilities and audit books and records.",
        "expected": {"obligations": [], "rights": ["inspect facilities", "audit books and records"], "conditions": [], "entities": ["Auditor"]}
    },
    {
        "id": "GEN_005", "category": "other_general",
        "text": "All notices shall be delivered in writing to Company headquarters.",
        "expected": {"obligations": ["delivered in writing to Company headquarters"], "rights": [], "conditions": [], "entities": ["Company"]}
    },

    # --- 10. termination_notice ---
    {
        "id": "TERM_001", "category": "termination_notice",
        "text": "Either party may terminate this Agreement without cause by giving at least thirty (30) days prior written notice to the other party.",
        "expected": {"notice_period": "thirty (30) days", "notice_given_by": "Either party", "termination_type": "without cause", "conditions": ["giving at least thirty (30) days prior written notice"], "effective_date": None}
    },
    {
        "id": "TERM_002", "category": "termination_notice",
        "text": "Employer may terminate employment immediately for cause upon written notice to Employee.",
        "expected": {"notice_period": "immediately", "notice_given_by": "Employer", "termination_type": "for cause", "conditions": ["upon written notice"], "effective_date": None}
    },
    {
        "id": "TERM_003", "category": "termination_notice",
        "text": "Either party may terminate this contract without cause upon sixty (60) days notice.",
        "expected": {"notice_period": "sixty (60) days", "notice_given_by": "Either party", "termination_type": "without cause", "conditions": ["upon sixty (60) days notice"], "effective_date": None}
    },
    {
        "id": "TERM_004", "category": "termination_notice",
        "text": "Company may terminate this Agreement for cause upon thirty (30) days written notice.",
        "expected": {"notice_period": "thirty (30) days", "notice_given_by": "Company", "termination_type": "for cause", "conditions": ["upon thirty (30) days written notice"], "effective_date": None}
    },
    {
        "id": "TERM_005", "category": "termination_notice",
        "text": "Either party may terminate without cause by providing ninety (90) days notice.",
        "expected": {"notice_period": "ninety (90) days", "notice_given_by": "Either party", "termination_type": "without cause", "conditions": ["providing ninety (90) days notice"], "effective_date": None}
    }
]

print(f"[✓] Loaded strict gold dataset with {len(BENCHMARK_DATASET)} ground-truth clauses.")

In [ ]:
# Execute Strict Benchmark Suite across all 50 Clauses using Updated Prompt with Few-Shot Examples
benchmark_results = []
category_scores = {cat: [] for cat in EXTRACTION_SCHEMAS.keys()}

print("=== RUNNING CLAUSEGUARD STRICT EXTRACTION BENCHMARK ===\n")

for sample in BENCHMARK_DATASET:
    cid = sample["id"]
    cat = sample["category"]
    text = sample["text"]
    exp = sample["expected"]
    schema_with_desc = EXTRACTION_SCHEMAS_WITH_DESCRIPTIONS[cat]
    
    # 1. NuExtract Inference using updated prompt rules and few-shot examples
    raw_out = extractor_runner.extract(text, schema_with_desc, category=cat)
    # 2. Validation with safe formatting-only repair
    valid, val_dict, val_errs = ExtractionValidator.validate_and_repair(raw_out, EXTRACTION_SCHEMAS[cat])
    # 3. Optional Normalization (attaches metadata while keeping raw legal text)
    norm_dict = Normalizer.normalize_extraction(cat, val_dict)
    # 4. Strict Field-by-Field Evaluation
    eval_report = BenchmarkEvaluator.evaluate_clause(exp, norm_dict)
    
    acc = eval_report["accuracy"]
    category_scores[cat].append(acc)
    
    record = {
        "id": cid,
        "category": cat,
        "text": text,
        "accuracy": acc,
        "expected": exp,
        "extracted": norm_dict,
        "eval_fields": eval_report["fields"]
    }
    benchmark_results.append(record)

print(f"[✓] Successfully executed benchmark across {len(benchmark_results)} strict gold clauses.")

In [ ]:
# Benchmark Results Summary Table
cat_summary = []
all_scores = []

for cat, scores in category_scores.items():
    avg_acc = (sum(scores) / len(scores)) * 100 if scores else 0.0
    all_scores.extend(scores)
    status_icon = "🟢 Excellent" if avg_acc >= 85 else "🟠 Moderate" if avg_acc >= 65 else "🔴 Major Miss"
    cat_summary.append({
        "Category": cat,
        "Family": CATEGORY_TO_FAMILY[cat],
        "Clauses Tested": len(scores),
        "Accuracy (%)": f"{avg_acc:.2f}%",
        "Status": status_icon
    })

summary_df = pd.DataFrame(cat_summary)
overall_accuracy = (sum(all_scores) / len(all_scores)) * 100 if all_scores else 0.0

print("===============================================================")
print(f"   STRICT CLAUSEGUARD BENCHMARK — CATEGORY METRICS             ")
print("===============================================================")
print(summary_df.to_string(index=False))
print("---------------------------------------------------------------")
print(f" OVERALL FIELD-LEVEL ACCURACY: {overall_accuracy:.2f}%")
print("===============================================================")

In [ ]:
# Discrepancies & Failure Report
print("\n=== FIELD DISCREPANCIES DIAGNOSTICS ===\n")
failure_count = 0

for rec in benchmark_results:
    if rec["accuracy"] < 1.0:
        failure_count += 1
        print(f"[{rec['id']}] Category: {rec['category']} | Accuracy: {rec['accuracy']*100:.1f}%")
        print(f"Clause Text: \"{rec['text']}\"")
        print("Field Discrepancies:")
        for fname, fmeta in rec["eval_fields"].items():
            if not fmeta["is_match"]:
                print(f"  ❌ Field '{fname}':")
                print(f"     Expected: {fmeta['expected']}")
                print(f"     Got:      {fmeta['got']}")
                print(f"     Detail:   {fmeta['detail']}")
        print("-" * 80)

if failure_count == 0:
    print("[✓] 100% strict field-level accuracy achieved across all 50 gold clauses.")
else:
    print(f"[!] Found {failure_count} clauses with discrepancies for benchmarking.")